In [0]:
1) Large file ingestion - Generators vs List

Interview Question

You are ingesting 50GB application log file in python and you only need to filter ERROR records before loading them into the Bronze layer. if you read entire file into a list, the job crashes due to memory issue. How would you solve this in python and why?

==> Use yield. it is a generator firstly And it will help to process one record at a time. It will not load entire record at a time. This will prevent the OOM issue.

In [0]:
# This will work like a streaming job inside of your batch job. During heavy ingestion of file it will also works. It is optimization technique

def read_logs(path):
    with open(path, 'r') as f: # open the particular path, r: read mode, so in read mode as a file f
        for line in f:
            yield f
error_logs = (log for log in read_logs("app.log") if "ERROR" in log)            

for log in error_logs:
    print(log)


2) Schema Builder Bug -- Mutable default Arguments

Interview Question:
You are writing a reusable Python function to dynamically build a list of columns for transformation. This function is called multiple times in the same pipeline run. However, columns from previous runs keep appearing unexpectedly. What is wrong in this code and how do you fix it?

In [0]:
# this is the predefined code and bydefault cols=[] is empty. 
def add_column(col,cols=[]):
    cols.append(col)
    return cols
add_column("order_id")
print(add_column("customer_id"))

# Now we dont want this op comes like this But I dont want like this ['order_id', 'customer_id']

In [0]:
# Solution
def add_column(col,cols=[]):
    if cols is None:
        cols = []
    cols.append(col)
    return cols
add_column("order_id")
print(add_column("customer_id"))


Interview Question: You load records into the Bronze layer, copy them, and enrich them for the Silver layer. After transformation, you notice Bronze data has also changed. Here is the code. Why is this happening

In [0]:
Shallow copy share the referances and DEEP copy isolates the data Across the layers. it will create a new set of things.

In [0]:
records = [{"id": 1, "tags": ["new"]}]
silver = records.copy()
silver[0]["tags"].append("processed")
silver

In [0]:
import copy
records = [{"id": 1, "tags": ["new"]}]
silver = copy.deepcopy(records)
silver[0]["tags"].append("processed")
silver

Interview Questions

You are building a Python pipeline framework where different transformation steps receive different parameters. How do you design a generic executor that support flexible inputs?


Answer Explanation:

*args and **krgs allow us to pass variable parameters dynamically, making pipelines configurable and reusuable. 

In [0]:
def run_step(step, *args, **kwargs):
    return step(*args, **kwargs)

def transform(value, multiplier=2):
    return value * multiplier

run_step(transform, 10, multiplier=5)

In [0]:
5) Retry failure- Idempotant ETL design:
Your Airflow task fails after writing partial data and then entries. How do you design your Python ETL so that retries don't create duplicate records?

= see this yt channel : https://youtu.be/yORFWmbnpKk